# Local Multimodal (Image+Text) Training
Train a local model that keeps image and text encoders separate and fuses them for multi-label classification.
The pipeline tolerates missing modalities per-sample.

In [2]:
import os, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from torchvision import transforms as T
from utils_fusion import (MMClientDataset, mm_collate, FusionClassifier, build_label_space, evaluate,
                          compute_pos_weight, train_one_epoch)

In [4]:
CLIENT_NUM = 12

# ---- Paths to your client's CSVs (update if needed) ----
BASE = f'C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_{CLIENT_NUM}'
# CSV_TRAIN_IMG = os.path.join(BASE, f'client_{CLIENT_NUM}_train_image.csv')
# CSV_TRAIN_TXT = os.path.join(BASE, f'client_{CLIENT_NUM}_train_text.csv')
# CSV_VAL_IMG   = os.path.join(BASE, f'client_{CLIENT_NUM}_val_image.csv')
# CSV_VAL_TXT   = os.path.join(BASE, f'client_{CLIENT_NUM}_val_text.csv')
# CSV_TEST_IMG  = os.path.join(BASE, f'client_{CLIENT_NUM}_test_image.csv')
# CSV_TEST_TXT  = os.path.join(BASE, f'client_{CLIENT_NUM}_test_text.csv')

# # Image roots per split (edit to your dataset layout)
# IMG_DIRS = {
#     'train': '../../rocov2/train_images/train',
#     'valid': '../../rocov2/valid_images/valid',
#     'test' : '../../rocov2/test_images/test',
# }

In [5]:
TEXT_MODEL = 'prajjwal1/bert-tiny'    # tiny for speed; swap to 'bert-base-uncased' etc.
MAX_LEN = 128
BATCH = 32
EPOCHS = 10
LR = 2e-4
THRESH = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _paths_for_client(cid, base=BASE):
    p = lambda split, mod: os.path.join(base, f"client_{cid}_{split}_{mod}.csv")
    return {
        "train_img": p("train","image"),
        "train_txt": p("train","text"),
        "val_img":   p("val","image"),
        "val_txt":   p("val","text"),
        "test_img":  p("test","image"),
        "test_txt":  p("test","text"),
    }

def train_client(client_id, base_dir=BASE, img_dirs=None,
                 text_model="prajjwal1/bert-tiny", max_len=128,
                 batch_size=32, epochs=10, lr=2e-4, thresh=0.5, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if img_dirs is None:
        img_dirs = {
            'train': '../../rocov2/train_images/train',
            'valid': '../../rocov2/valid_images/valid',
            'test' : '../../rocov2/test_images/test',
        }

    # --- 파일 경로 ---
    CSV_TRAIN_IMG = os.path.join(base_dir, f"client_{client_id}_train_image.csv")
    CSV_TRAIN_TXT = os.path.join(base_dir, f"client_{client_id}_train_text.csv")
    CSV_VAL_IMG   = os.path.join(base_dir, f"client_{client_id}_val_image.csv")
    CSV_VAL_TXT   = os.path.join(base_dir, f"client_{client_id}_val_text.csv")
    CSV_TEST_IMG  = os.path.join(base_dir, f"client_{client_id}_test_image.csv")
    CSV_TEST_TXT  = os.path.join(base_dir, f"client_{client_id}_test_text.csv")

    # --- 데이터 로딩 ---
    df_tr_img = pd.read_csv(CSV_TRAIN_IMG)
    df_tr_txt = pd.read_csv(CSV_TRAIN_TXT)
    df_va_img = pd.read_csv(CSV_VAL_IMG)
    df_va_txt = pd.read_csv(CSV_VAL_TXT)
    df_te_img = pd.read_csv(CSV_TEST_IMG)
    df_te_txt = pd.read_csv(CSV_TEST_TXT)

    label2idx, idx2label = build_label_space(df_tr_img, df_tr_txt, df_va_img, df_va_txt, df_te_img, df_te_txt)
    num_classes = len(label2idx)

    tokenizer = AutoTokenizer.from_pretrained(text_model)
    img_tf = T.Compose([T.Resize((224,224)), T.ToTensor()])

    ds_tr = MMClientDataset(df_tr_img, df_tr_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)
    ds_va = MMClientDataset(df_va_img, df_va_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)
    ds_te = MMClientDataset(df_te_img, df_te_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)

    collate = lambda batch: mm_collate(batch, tokenizer, max_len=max_len)
    tr_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate)
    va_loader = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate)
    te_loader = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate)

    # --- 모델 ---
    model = FusionClassifier(num_classes=num_classes, text_model_name=text_model).to(device)
    pos_w = compute_pos_weight(tr_loader, device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w) if pos_w is not None else torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # --- 학습 ---
    best = {"f1_micro": -1.0}
    best_state = None
    for ep in range(1, epochs+1):
        tl = train_one_epoch(model, tr_loader, optimizer, criterion, device, log_every=50)
        met = evaluate(model, va_loader, device, threshold=thresh)
        print(f"[Client {client_id} | Epoch {ep}] train_loss={tl:.4f} | val:", met)
        if met["f1_micro"] > best["f1_micro"]:
            best = met
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            out_path = os.path.join(base_dir, f"client_{client_id}_fusion_best.pt")
            torch.save(best_state, out_path)
            print(f"[Client {client_id}] BEST updated → {out_path}")

    # --- 테스트 ---
    if best_state is not None:
        model.load_state_dict(best_state)
    test_met = evaluate(model, te_loader, device, threshold=thresh)
    print(f"[Client {client_id}] Test:", test_met)
    return test_met


def train_client_singlemod(client_id, modality="image",
                           base_dir="/mnt/data", img_dirs=None,
                           text_model="prajjwal1/bert-tiny",
                           max_len=128, batch_size=32, epochs=10, lr=2e-4,
                           thresh=0.5, device=None):
    """
    client_id : int
    modality : "image" | "text"
    """

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if img_dirs is None:
        img_dirs = {
            'train': '../../rocov2/train_images/train',
            'valid': '../../rocov2/valid_images/valid',
            'test' : '../../rocov2/test_images/test',
        }

    # --- CSV 경로 ---
    CSV_TRAIN_IMG = os.path.join(base_dir, f"client_{client_id}_train_image.csv")
    CSV_TRAIN_TXT = os.path.join(base_dir, f"client_{client_id}_train_text.csv")
    CSV_VAL_IMG   = os.path.join(base_dir, f"client_{client_id}_val_image.csv")
    CSV_VAL_TXT   = os.path.join(base_dir, f"client_{client_id}_val_text.csv")
    CSV_TEST_IMG  = os.path.join(base_dir, f"client_{client_id}_test_image.csv")
    CSV_TEST_TXT  = os.path.join(base_dir, f"client_{client_id}_test_text.csv")

    # --- 데이터 로딩 ---
    df_tr_img = pd.read_csv(CSV_TRAIN_IMG) if os.path.exists(CSV_TRAIN_IMG) else None
    df_tr_txt = pd.read_csv(CSV_TRAIN_TXT) if os.path.exists(CSV_TRAIN_TXT) else None
    df_va_img = pd.read_csv(CSV_VAL_IMG) if os.path.exists(CSV_VAL_IMG) else None
    df_va_txt = pd.read_csv(CSV_VAL_TXT) if os.path.exists(CSV_VAL_TXT) else None
    df_te_img = pd.read_csv(CSV_TEST_IMG) if os.path.exists(CSV_TEST_IMG) else None
    df_te_txt = pd.read_csv(CSV_TEST_TXT) if os.path.exists(CSV_TEST_TXT) else None

    # --- label space ---
    label2idx, idx2label = build_label_space(df_tr_img, df_tr_txt, df_va_img, df_va_txt, df_te_img, df_te_txt)
    num_classes = len(label2idx)

    tokenizer = AutoTokenizer.from_pretrained(text_model)
    img_tf = T.Compose([T.Resize((224,224)), T.ToTensor()])

    # --- Dataset (modality 제한) ---
    if modality == "image":
        ds_tr = MMClientDataset(df_tr_img, None, label2idx, img_dirs, transform=img_tf, max_len=max_len)
        ds_va = MMClientDataset(df_va_img, None, label2idx, img_dirs, transform=img_tf, max_len=max_len)
        ds_te = MMClientDataset(df_te_img, None, label2idx, img_dirs, transform=img_tf, max_len=max_len)
    elif modality == "text":
        ds_tr = MMClientDataset(None, df_tr_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)
        ds_va = MMClientDataset(None, df_va_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)
        ds_te = MMClientDataset(None, df_te_txt, label2idx, img_dirs, transform=img_tf, max_len=max_len)
    else:
        raise ValueError("modality must be 'image' or 'text'")

    collate = lambda batch: mm_collate(batch, tokenizer, max_len=max_len)
    tr_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate)
    va_loader = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate)
    te_loader = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate)

    # --- Model ---
    model = FusionClassifier(num_classes=num_classes, text_model_name=text_model,
                             image_trainable=True, text_trainable=True).to(device)
    pos_w = compute_pos_weight(tr_loader, device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w) if pos_w is not None else torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # --- Train ---
    best = {"f1_micro": -1.0}
    best_state = None
    for ep in range(1, epochs+1):
        tl = train_one_epoch(model, tr_loader, optimizer, criterion, device, log_every=50)
        met = evaluate(model, va_loader, device, threshold=thresh)
        print(f"[Client {client_id} | Epoch {ep}] train_loss={tl:.4f} | val:", met)
        if met["f1_micro"] > best["f1_micro"]:
            best = met
            best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            out_path = os.path.join(base_dir, f"client_{client_id}_{modality}_best.pt")
            torch.save(best_state, out_path)
            print(f"[Client {client_id}] BEST updated → {out_path}")

    # --- Test ---
    if best_state is not None:
        model.load_state_dict(best_state)
    test_met = evaluate(model, te_loader, device, threshold=thresh)
    print(f"[Client {client_id}] Test:", test_met)
    return test_met


#
# df_tr_img = pd.read_csv(CSV_TRAIN_IMG)
# df_tr_txt = pd.read_csv(CSV_TRAIN_TXT)
# df_va_img = pd.read_csv(CSV_VAL_IMG)
# df_va_txt = pd.read_csv(CSV_VAL_TXT)
# df_te_img = pd.read_csv(CSV_TEST_IMG)
# df_te_txt = pd.read_csv(CSV_TEST_TXT)
#
# label2idx, idx2label = build_label_space(df_tr_img, df_tr_txt, df_va_img, df_va_txt, df_te_img, df_te_txt)
# NUM_CLASSES = len(label2idx)
# print('Num classes:', NUM_CLASSES)
#
# tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
# img_tf = T.Compose([T.Resize((224,224)), T.ToTensor()])
#
# ds_tr = MMClientDataset(df_tr_img, df_tr_txt, label2idx, IMG_DIRS, transform=img_tf, max_len=MAX_LEN)
# ds_va = MMClientDataset(df_va_img, df_va_txt, label2idx, IMG_DIRS, transform=img_tf, max_len=MAX_LEN)
# ds_te = MMClientDataset(df_te_img, df_te_txt, label2idx, IMG_DIRS, transform=img_tf, max_len=MAX_LEN)
#
# collate = lambda batch: mm_collate(batch, tokenizer, max_len=MAX_LEN)
#
# tr_loader = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=0, collate_fn=collate)
# va_loader = DataLoader(ds_va, batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate)
# te_loader = DataLoader(ds_te, batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate)
#
# b = next(iter(tr_loader))
# print("pixel_values", b["pixel_values"].shape)    # 기대: (B,3,224,224)
# print("input_ids", b["input_ids"].shape)          # 기대: (B,L)
# print("attention_mask", b["attention_mask"].shape)# 기대: (B,L)
# print("img_mask", b["img_mask"].shape)            # 기대: (B,1)
# print("txt_mask", b["txt_mask"].shape)            # 기대: (B,1)
# print("labels", b["labels"].shape)                # 기대: (B,C)
#
# model = FusionClassifier(num_classes=NUM_CLASSES, text_model_name=TEXT_MODEL, image_trainable=True, text_trainable=True).to(DEVICE)
# pos_w = compute_pos_weight(tr_loader, DEVICE)
# criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w) if pos_w is not None else torch.nn.BCEWithLogitsLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


In [43]:
# best = {'f1_micro': -1.0}
# best_state = None
# for ep in range(1, EPOCHS+1):
#     try:
#         tl = train_one_epoch(model, tr_loader, optimizer, criterion, DEVICE, log_every=50)
#         met = evaluate(model, va_loader, DEVICE, threshold=THRESH)
#         print(f'[EPOCH {ep}] train_loss={tl:.4f} | val:', met)
#         if met['f1_micro'] > best['f1_micro']:
#             best = met
#             best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
#             torch.save(best_state, os.path.join(BASE, f'client_{CLIENT_NUM}_local_fusion_best.pt'))
#             print('[BEST] updated', best)
#     except Exception as e:
#         import traceback
#         print("[EXC] error in epoch", ep, e)
#         traceback.print_exc()
#         break   # → 우선 멈추고 원인을 확인
# print('Best val:', best)

2025-08-31 18:24:32,817 INFO: [HB train] i=50/157 loss=0.6104
2025-08-31 18:24:34,133 INFO: [HB train] i=100/157 loss=0.5113
2025-08-31 18:24:35,445 INFO: [HB train] i=150/157 loss=0.4788
[EPOCH 1] train_loss=0.4758 | val: {'f1_micro': 0.0919225503618228, 'f1_macro': 0.013462841065288798, 'auc_macro': 0.7071493778516031}
[BEST] updated {'f1_micro': 0.0919225503618228, 'f1_macro': 0.013462841065288798, 'auc_macro': 0.7071493778516031}
2025-08-31 18:24:37,574 INFO: [HB train] i=50/157 loss=0.3761
2025-08-31 18:24:38,896 INFO: [HB train] i=100/157 loss=0.3596
2025-08-31 18:24:40,218 INFO: [HB train] i=150/157 loss=0.3417
[EPOCH 2] train_loss=0.3384 | val: {'f1_micro': 0.2199034418314904, 'f1_macro': 0.03462040238798097, 'auc_macro': 0.8385003662129376}
[BEST] updated {'f1_micro': 0.2199034418314904, 'f1_macro': 0.03462040238798097, 'auc_macro': 0.8385003662129376}
2025-08-31 18:24:42,334 INFO: [HB train] i=50/157 loss=0.2464
2025-08-31 18:24:43,694 INFO: [HB train] i=100/157 loss=0.2401
2

In [44]:
# if best_state is not None:
#     model.load_state_dict(best_state)
# test_met = evaluate(model, te_loader, DEVICE, threshold=THRESH)
# print('Test:', test_met)


Test: {'f1_micro': 0.47819767441860467, 'f1_macro': 0.24536214766841813, 'auc_macro': 0.9686790067737145}


In [48]:
results = {}
for cid in range(13, 26):   # client_13 ~ client_25
    print("="*20, f"Client {cid}", "="*20)
    results[cid] = train_client(cid, base_dir=f'C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_{cid}')


==================== Client 13 ====================


C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 18:31:34,845 INFO: [HB train] i=50/157 loss=0.5878
2025-08-31 18:31:36,175 INFO: [HB train] i=100/157 loss=0.4989
2025-08-31 18:31:37,521 INFO: [HB train] i=150/157 loss=0.4651
[Client 13 | Epoch 1] train_loss=0.4624 | val: {'f1_micro': 0.10308488216133359, 'f1_macro': 0.013550143400469734, 'auc_macro': 0.7135466571671504}
[Client 13] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_13\client_13_fusion_best.pt
2025-08-31 18:31:39,701 INFO: [HB train] i=50/157 loss=0.3627
2025-08-31 18:31:41,056 INFO: [HB train] i=100/157 loss=0.3445
2025-08-31 18:31:42,412 INFO: [HB train] i=150/157 loss=0.3273
[Client 13 | Epoch 2] train_loss=0.3256 | val: {'f1_micro': 0.22119402985074627, 'f1_macro': 0.03596255333926061, 'auc_macro': 0.86340308794674}
[Client 13] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_13\client_13_fusion_best.pt
2025-08-31 18:31:44,556 INFO: [HB train] i=50/157 loss=0.2335
2025-08-31 18:31:45,892 I

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 18:32:26,192 INFO: [HB train] i=50/157 loss=0.5989
2025-08-31 18:32:27,510 INFO: [HB train] i=100/157 loss=0.5024
2025-08-31 18:32:28,837 INFO: [HB train] i=150/157 loss=0.4683
[Client 14 | Epoch 1] train_loss=0.4657 | val: {'f1_micro': 0.11303861219921657, 'f1_macro': 0.016766727833030213, 'auc_macro': 0.7342968107434272}
[Client 14] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_14\client_14_fusion_best.pt
2025-08-31 18:32:30,982 INFO: [HB train] i=50/157 loss=0.3684
2025-08-31 18:32:32,325 INFO: [HB train] i=100/157 loss=0.3519
2025-08-31 18:32:33,676 INFO: [HB train] i=150/157 loss=0.3345
[Client 14 | Epoch 2] train_loss=0.3319 | val: {'f1_micro': 0.25753424657534246, 'f1_macro': 0.05097324242333055, 'auc_macro': 0.8672303088377498}
[Client 14] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_14\client_14_fusion_best.pt
2025-08-31 18:32:35,777 INFO: [HB train] i=50/157 loss=0.2410
2025-08-31 18:32:37,108

...

2025-08-31 18:40:13,351 INFO: [HB train] i=50/157 loss=0.5810
2025-08-31 18:40:14,729 INFO: [HB train] i=100/157 loss=0.5010
2025-08-31 18:40:16,060 INFO: [HB train] i=150/157 loss=0.4631
[Client 23 | Epoch 1] train_loss=0.4585 | val: {'f1_micro': 0.1152, 'f1_macro': 0.01890114402649593, 'auc_macro': 0.7222348477604946}
[Client 23] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_23\client_23_fusion_best.pt
2025-08-31 18:40:18,283 INFO: [HB train] i=50/157 loss=0.3500
2025-08-31 18:40:19,657 INFO: [HB train] i=100/157 loss=0.3347
2025-08-31 18:40:21,048 INFO: [HB train] i=150/157 loss=0.3178
[Client 23 | Epoch 2] train_loss=0.3153 | val: {'f1_micro': 0.24601437949359176, 'f1_macro': 0.047706652612674524, 'auc_macro': 0.8674432083831471}
[Client 23] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_23\client_23_fusion_best.pt
2025-08-31 18:40:23,233 INFO: [HB train] i=50/157 loss=0.2399
2025-08-31 18:40:24,614 INFO: [HB tr

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 18:41:05,867 INFO: [HB train] i=50/157 loss=0.5863
2025-08-31 18:41:07,208 INFO: [HB train] i=100/157 loss=0.5008
2025-08-31 18:41:08,634 INFO: [HB train] i=150/157 loss=0.4666
[Client 24 | Epoch 1] train_loss=0.4642 | val: {'f1_micro': 0.09944422389837237, 'f1_macro': 0.01080208277298429, 'auc_macro': 0.7099345644206337}
[Client 24] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_24\client_24_fusion_best.pt
2025-08-31 18:41:10,766 INFO: [HB train] i=50/157 loss=0.3601
2025-08-31 18:41:12,203 INFO: [HB train] i=100/157 loss=0.3508
2025-08-31 18:41:13,554 INFO: [HB train] i=150/157 loss=0.3411
[Client 24 | Epoch 2] train_loss=0.3389 | val: {'f1_micro': 0.21424278846153846, 'f1_macro': 0.03326147938639645, 'auc_macro': 0.8404248087871594}
[Client 24] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_24\client_24_fusion_best.pt
2025-08-31 18:41:15,672 INFO: [HB train] i=50/157 loss=0.2523
2025-08-31 18:41:17,066 

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 18:41:58,353 INFO: [HB train] i=50/157 loss=0.5830
2025-08-31 18:41:59,699 INFO: [HB train] i=100/157 loss=0.4970
2025-08-31 18:42:01,050 INFO: [HB train] i=150/157 loss=0.4623
[Client 25 | Epoch 1] train_loss=0.4585 | val: {'f1_micro': 0.09121652499771501, 'f1_macro': 0.013600039788163033, 'auc_macro': 0.7162108349657318}
[Client 25] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_25\client_25_fusion_best.pt
2025-08-31 18:42:03,195 INFO: [HB train] i=50/157 loss=0.3517
2025-08-31 18:42:04,709 INFO: [HB train] i=100/157 loss=0.3402
2025-08-31 18:42:06,251 INFO: [HB train] i=150/157 loss=0.3215
[Client 25 | Epoch 2] train_loss=0.3189 | val: {'f1_micro': 0.27469026548672565, 'f1_macro': 0.06457266350764185, 'auc_macro': 0.864426765557968}
[Client 25] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_25\client_25_fusion_best.pt
2025-08-31 18:42:08,478 INFO: [HB train] i=50/157 loss=0.2182
2025-08-31 18:42:09,921 

In [6]:
results = {}

# 26,27번 클라이언트는 이미지 전용
for cid in [26, 27]:
    results[cid] = train_client_singlemod(cid, base_dir=f'C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_{cid}', modality="image")

# 28,29번 클라이언트는 텍스트 전용
for cid in [28, 29]:
    results[cid] = train_client_singlemod(cid, base_dir=f'C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_{cid}', modality="text")

print("Final results:", results)


C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 19:07:19,752 INFO: [HB train] i=50/157 loss=0.7998
2025-08-31 19:07:20,963 INFO: [HB train] i=100/157 loss=0.7877
2025-08-31 19:07:22,188 INFO: [HB train] i=150/157 loss=0.7699
[Client 26 | Epoch 1] train_loss=0.7670 | val: {'f1_micro': 0.05294803912770349, 'f1_macro': 0.0024666441813377534, 'auc_macro': 0.5000231209205239}
[Client 26] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_26\client_26_image_best.pt
2025-08-31 19:07:24,241 INFO: [HB train] i=50/157 loss=0.6704
2025-08-31 19:07:25,463 INFO: [HB train] i=100/157 loss=0.6403
2025-08-31 19:07:26,679 INFO: [HB train] i=150/157 loss=0.6123
[Client 26 | Epoch 2] train_loss=0.6084 | val: {'f1_micro': 0.07195856082878342, 'f1_macro': 0.0021134965651824996, 'auc_macro': 0.4997207259996015}
[Client 26] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_26\client_26_image_best.pt
2025-08-31 19:07:28,627 INFO: [HB train] i=50/157 loss=0.5103
2025-08-31 19:07:29,85

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 19:08:06,205 INFO: [HB train] i=50/157 loss=0.8052
2025-08-31 19:08:07,420 INFO: [HB train] i=100/157 loss=0.7915
2025-08-31 19:08:08,613 INFO: [HB train] i=150/157 loss=0.7749
[Client 27 | Epoch 1] train_loss=0.7720 | val: {'f1_micro': 0.06609547123623011, 'f1_macro': 0.002227770407384029, 'auc_macro': 0.5000685703550339}
[Client 27] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_27\client_27_image_best.pt
2025-08-31 19:08:10,486 INFO: [HB train] i=50/157 loss=0.6821
2025-08-31 19:08:11,693 INFO: [HB train] i=100/157 loss=0.6536
2025-08-31 19:08:12,993 INFO: [HB train] i=150/157 loss=0.6255
[Client 27 | Epoch 2] train_loss=0.6218 | val: {'f1_micro': 0.06529006882989184, 'f1_macro': 0.0027502935843694482, 'auc_macro': 0.5001139771835262}
2025-08-31 19:08:14,849 INFO: [HB train] i=50/157 loss=0.5177
2025-08-31 19:08:16,024 INFO: [HB train] i=100/157 loss=0.5010
2025-08-31 19:08:17,213 INFO: [HB train] i=150/157 loss=0.4853
[Client 27 | Epoch

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 19:08:51,964 INFO: [HB train] i=50/157 loss=0.5965
2025-08-31 19:08:53,240 INFO: [HB train] i=100/157 loss=0.5128
2025-08-31 19:08:54,518 INFO: [HB train] i=150/157 loss=0.4750
[Client 28 | Epoch 1] train_loss=0.4707 | val: {'f1_micro': 0.08657198741389574, 'f1_macro': 0.013552311789084734, 'auc_macro': 0.7088817919499182}
[Client 28] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_28\client_28_text_best.pt
2025-08-31 19:08:56,514 INFO: [HB train] i=50/157 loss=0.3744
2025-08-31 19:08:57,768 INFO: [HB train] i=100/157 loss=0.3547
2025-08-31 19:08:59,039 INFO: [HB train] i=150/157 loss=0.3374
[Client 28 | Epoch 2] train_loss=0.3360 | val: {'f1_micro': 0.22507528926929782, 'f1_macro': 0.04156776870841422, 'auc_macro': 0.8475548628532278}
[Client 28] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_28\client_28_text_best.pt
2025-08-31 19:09:01,055 INFO: [HB train] i=50/157 loss=0.2421
2025-08-31 19:09:02,333 INF

C:\Users\user\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


2025-08-31 19:09:40,152 INFO: [HB train] i=50/157 loss=0.5847
2025-08-31 19:09:41,431 INFO: [HB train] i=100/157 loss=0.4958
2025-08-31 19:09:42,686 INFO: [HB train] i=150/157 loss=0.4624
[Client 29 | Epoch 1] train_loss=0.4578 | val: {'f1_micro': 0.10599690196946226, 'f1_macro': 0.01163649615667283, 'auc_macro': 0.7353812074287719}
[Client 29] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_29\client_29_text_best.pt
2025-08-31 19:09:44,672 INFO: [HB train] i=50/157 loss=0.3539
2025-08-31 19:09:45,952 INFO: [HB train] i=100/157 loss=0.3455
2025-08-31 19:09:47,236 INFO: [HB train] i=150/157 loss=0.3266
[Client 29 | Epoch 2] train_loss=0.3237 | val: {'f1_micro': 0.21663554087056458, 'f1_macro': 0.03861166234559994, 'auc_macro': 0.8709083563548964}
[Client 29] BEST updated → C:/Users/user/PycharmProjects/ROCO/archive/rocov2/_prepared/client_29\client_29_text_best.pt
2025-08-31 19:09:49,223 INFO: [HB train] i=50/157 loss=0.2515
2025-08-31 19:09:50,488 INFO